In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pycontrails import Fleet

In [ ]:
# TODO run one cell (depending on the metric) and then plot the table (same as in table-results)

In [2]:
# EAGWP100 (-13.3%)
dff = pd.read_parquet("../data/filed_trajectories_EAGWP100.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised_trajectories_EAGWP100.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


In [7]:
# ATR100 (-13.9%)
dff = pd.read_parquet("../data/filed_trajectories_ATR100.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised_trajectories_ATR100.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


In [12]:
# EAGWP20 (-21.1%)
dff = pd.read_parquet("../data/filed_trajectories_EAGWP20.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised_trajectories_EAGWP20.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


In [35]:
# # EAGWP100 (-13.5%) (without CO2eq; so no, it doesn't play a role)
# dff = pd.read_parquet("./data/dff_matthes_2025_dahlmann_2025_EAGWP100_[150, 350]_False.parquet")
# fleetf = Fleet(data=dff)
# print(f"Fleet contains {fleetf.n_flights} flights")
#
# dfo = pd.read_parquet("./data/dfo_matthes_2025_dahlmann_2025_EAGWP100_[150, 350]_False.parquet")
# fleeto = Fleet(data=dfo)
# print(f"Fleet contains {fleeto.n_flights} flights")

In [36]:
# # ATR20 (-22.3%), another try
# dff = pd.read_parquet("./data/dff_matthes_2025_dahlmann_2025_ATR20_[150, 350]_False.parquet")
# fleetf = Fleet(data=dff)
# print(f"Fleet contains {fleetf.n_flights} flights")
#
# dfo = pd.read_parquet("./data/dfo_matthes_2025_dahlmann_2025_ATR20_[150, 350]_False.parquet")
# fleeto = Fleet(data=dfo)
# print(f"Fleet contains {fleeto.n_flights} flights")

In [13]:
# TODO fuel burn? nox? depends on if I want to have cruise only
# cols = ["fuel_burn", "nox", "NOx", "O3", "CH4", "H2O"]
cols = ["NOx", "O3", "CH4", "H2O"]

from cane.utils import mask_by_marker

mask_by_marker(fleetf, cols)
mask_by_marker(fleeto, cols)

dff = fleetf.dataframe
dfo = fleeto.dataframe

from cane.utils import mask_by_validity_range

bounds = [150, 350]
mask_by_validity_range(fleetf, cols, bounds)
mask_by_validity_range(fleeto, cols, bounds)

dff = fleetf.dataframe
dfo = fleeto.dataframe

Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O


In [14]:
dff["CO2_CoCiP"] = dff["CO2"] + dff["CoCiP"]
dfo["CO2_CoCiP"] = dfo["CO2"] + dfo["CoCiP"]

dff["Total"] = dff["CO2"] + dff["CoCiP"] + dff["NOx"] + dff["H2O"]
dfo["Total"] = dfo["CO2"] + dfo["CoCiP"] + dfo["NOx"] + dfo["H2O"]

In [15]:
from cane.utils import df_diff

# create aggregates (total)
org_sum = (
    dff[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP"]]
    .groupby(["flight_id"]).sum().reset_index()
)
opt_sum = (
    dfo[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP"]]
    .groupby(["flight_id"]).sum().reset_index()
)
my_diff = df_diff(org_sum, opt_sum, on=["flight_id"], keep_originals=True)
my_diff = pd.merge(
    my_diff,
    dff.groupby("flight_id").day.first(),
    on="flight_id",
)

In [6]:
# the next rows are for the summary (not for the table later)
my_diff_egwp100 = my_diff.copy()

In [11]:
my_diff_atr100 = my_diff.copy()

In [16]:
my_diff_egwp20 = my_diff.copy()

In [17]:
print("# flights:", my_diff_egwp100.shape[0], my_diff_atr100.shape[0], my_diff_egwp20.shape[0])
dfs = [my_diff_egwp100, my_diff_atr100, my_diff_egwp20]

# flights: 4112 4112 4112


# July 2026

In [19]:
# July 8
def delta(before: float, after: float) -> float:
    # in %
    if before == 0:
        raise ValueError("'before' cannot be zero — percent change is undefined.")
    return round((after - before) / before * 100, 1)

In [22]:
print("EAGWP100", round(my_diff_egwp100["Total_diff"].sum() / 4112 / 1e3, 3), delta(my_diff_egwp100["Total_filed"].sum(), my_diff_egwp100["Total_optimised"].sum()))

EAGWP100 -25.161 -11.1


In [23]:
print("ATR100", round(my_diff_atr100["Total_diff"].sum() / 4112 / 1e3, 3), delta(my_diff_atr100["Total_filed"].sum(), my_diff_atr100["Total_optimised"].sum()))

ATR100 -33.095 -12.4


In [24]:
print("EGWP20", round(my_diff_egwp20["Total_diff"].sum() / 4112 / 1e3, 3), delta(my_diff_egwp20["Total_filed"].sum(), my_diff_egwp20["Total_optimised"].sum()))

EGWP20 -95.258 -16.2


# old

In [18]:
level1 = np.array([df.query("CO2_CoCiP_diff < 0").shape[0] for df in dfs])
print(level1)
print(level1.min(), "--", level1.max())
print(round(level1.min() / 4112 * 100, 1), "--", round(level1.max() / 4112 * 100, 1))

[3503 3587 3572]
3503 -- 3587
85.2 -- 87.2


In [19]:
level2 = np.array([df.query("CO2_CoCiP_diff < 0 and Total_diff < 0").shape[0] for df in dfs])
print(level2)
print(level2.min(), "--", level2.max())
print(round(level2.min() / 4112 * 100, 1), "--", round(level2.max() / 4112 * 100, 1))

[3410 3563 3483]
3410 -- 3563
82.9 -- 86.6


In [18]:
level3 = np.array([df.query("Total_diff < 0").shape[0] for df in dfs])
print(level3)
print(level3.min(), "--", level3.max())
print(round(level3.min() / 4112 * 100, 1), "--", round(level3.max() / 4112 * 100, 1))

[3455 3485 3547]
3455 -- 3547
84.0 -- 86.3


In [41]:
cols = ["fuel_burn_diff", "nox_diff", "co2_diff", "CO2_diff", "NOx_diff", "CoCiP_diff", "H2O_diff", "CO2_CoCiP_diff",
        "Total_diff"]

tmp = my_diff[cols].sum().rename("diff_per_flight") / my_diff.shape[0]  # changes in [kg] per rerouted flight
tmp = tmp.reset_index()
# tmp["diff_per_flight"] = tmp["diff_per_flight"]  #.round(2)

In [42]:
cols = ["fuel_burn", "nox", "co2", "CO2", "NOx", "CoCiP", "H2O", "CO2_CoCiP", "Total"]

rels = my_diff[[f"{col}_diff" for col in cols]].sum().values / my_diff[
    [f"{col}_filed" for col in cols]].sum().values  # relative change

tmp["reldiff_per_flight"] = rels
tmp
# for i, r in tmp.iterrows():
#     print(r["index"], round(r.diff_per_flight, 2), f"{round(r.reldiff_per_flight * 100, 2)}%", sep="\t")

,index,diff_per_flight,reldiff_per_flight
0,fuel_burn_diff,357.330335,0.011945
1,nox_diff,12.549362,0.025050
2,co2_diff,1128.806529,0.011945
3,CO2_diff,1128.806529,0.011945
4,NOx_diff,1566.055677,0.012430
5,CoCiP_diff,-99164.154278,-0.435171
6,H2O_diff,-123.313753,-0.012135
7,CO2_CoCiP_diff,-98061.639727,-0.304187
8,Total_diff,-96618.897804,-0.210716


# June 2026 (for discussion)

In [25]:
agwp100_gaillot = 8.8e-14
agwp20_gaillot = 2.39e-14

agwp100_dahlmann = 7.435e-14
agwp20_dahlmann = 2.416e-14

print((1 - agwp100_dahlmann / agwp100_gaillot) * 100, "% lower")
print((agwp20_dahlmann / agwp20_gaillot - 1) * 100, "% higher")

15.51136363636363 % lower
1.087866108786617 % higher
